<a href="https://colab.research.google.com/github/kirthankulkarni-bit/assip-ML-stress-monitoring/blob/main/multiple_subjects.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os
import pickle
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, filtfilt, find_peaks
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# mount drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# slight variation of previous data filtering code but in one function

def process_subject(subject_id, file_path):
    print(f"processing {subject_id}")

    # load new data
    with open(file_path, 'rb') as f:
        subject_data = pickle.load(f, encoding='latin1')

    raw_eda = subject_data['signal']['chest']['EDA'].flatten()
    raw_bvp = subject_data['signal']['chest']['BVP'].flatten()
    labels = subject_data['label'].flatten()

    # sampling rate constants
    fs_eda, fs_bvp, fs_label = 4, 64, 700

    # EDA Filters (1hz lowpass to clean, 0.05hz highpass for phasic)
    b_low, a_low = butter(4, 1.0 / (0.5 * fs_eda), btype='low')
    clean_eda = filtfilt(b_low, a_low, raw_eda)
    b_high, a_high = butter(4, 0.05 / (0.5 * fs_eda), btype='high')
    phasic_eda = filtfilt(b_high, a_high, clean_eda)

    # BVP Filter (0.5 - 8hz bandpass)
    b_band, a_band = butter(4, [0.5 / (0.5 * fs_bvp), 8.0 / (0.5 * fs_bvp)], btype='band')
    clean_bvp = filtfilt(b_band, a_band, raw_bvp)

    # Window constants
    window_size = 60
    step_size = 30
    total_sec = len(labels) // fs_label

    features_list = []

    # sliding window loop
    for start in range(0, total_sec - window_size, step_size):
        eda_slice = phasic_eda[start * fs_eda : (start + window_size) * fs_eda]
        bvp_slice = clean_bvp[start * fs_bvp : (start + window_size) * fs_bvp]
        label_slice = labels[start * fs_label : (start + window_size) * fs_label]

        majority_label = np.bincount(label_slice).argmax()

        if majority_label in [1, 2]:
            bvp_peaks, _ = find_peaks(bvp_slice, distance=32)

            features_list.append({
                'subject': subject_id,
                'eda_mean': np.mean(eda_slice),
                'eda_std': np.std(eda_slice),
                'eda_max': np.max(eda_slice),
                'eda_min': np.min(eda_slice),
                'bvp_hr_bpm': len(bvp_peaks),
                'bvp_std': np.std(bvp_slice),
                'label': majority_label
            })

    df = pd.DataFrame(features_list)
    print(f"done with {subject_id}; extracted {len(df)} windows")
    return df